# Fine-Tuning Textbook Comparison

## Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_poisson_deviance
from datasets import Dataset
import statsmodels.api as sm
import matplotlib.pyplot as plt


## Raw Data GLM

In [2]:
# ==========================================
# DATA LOADING & PREPROCESSING
# ==========================================
print("Loading freMTPL2freq dataset...")
dataset = fetch_openml(data_id=41214, as_frame=True)
dat = dataset.frame

# Clean basic types first
dat['ClaimNb'] = pd.to_numeric(dat['ClaimNb'])
dat['Exposure'] = pd.to_numeric(dat['Exposure'])
dat['Exposure'] = dat['Exposure'].clip(upper=1.0)
dat['Frequency'] = dat['ClaimNb'] / dat['Exposure']

# Perform same mapping and engineering as Statistical Foundations for Actuarial Learning 5.2.4.
dat = pd.get_dummies(dat, columns=['VehGas'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehBrand'],drop_first=True)
dat = pd.get_dummies(dat, columns=['Region'],drop_first=True)
area_remapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6
}
dat["Area"] = dat["Area"].map(area_remapping)

dat['VehPowerGLM'] = pd.Categorical(np.minimum(dat['VehPower'], 9))

dat['VehAgeGLM'] = pd.cut(
    dat['VehAge'],
    bins=[0, 5, 12, 101],
    labels=["0-5", "6-12", "12+"],
    include_lowest=True
)

dat['DrivAgeGLM'] = pd.cut(
    dat['DrivAge'],
    bins=[18, 20, 25, 30, 40, 50, 70, 101],
    labels=["18-20", "21-25", "26-30", "31-40", "41-50", "51-70", "71+"],
    include_lowest=True
)

dat['BonusMalusGLM'] = np.minimum(dat['BonusMalus'], 150)

dat['DensityGLM'] = np.log(dat["Density"].astype(float))

dat = pd.get_dummies(dat, columns=['DrivAgeGLM'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehAgeGLM'],drop_first=True)
dat = pd.get_dummies(dat, columns=['VehPowerGLM'],drop_first=True)

dat = dat.drop(["VehPower", "VehAge", "DrivAge", "BonusMalus","Density"], axis = 1)
dat = dat.astype('float32')

Loading freMTPL2freq dataset...


In [3]:
# Load the split indices
df_splits = pd.read_csv('freMTPL2freq_split_indices.csv')

# Ensure IDpol is the same type in both dataframes for a clean merge
dat['IDpol'] = dat['IDpol'].astype(int)
df_splits['IDpol'] = df_splits['IDpol'].astype(int)

# Merge the dataset with the split indicators
# We use a left join to keep the original data rows
df_merged = dat.merge(df_splits, on='IDpol', how='left')

# Create the subsets based on the indicator columns
train_df = df_merged[df_merged['is_train'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
test_df = df_merged[df_merged['is_test'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
finetune_df = df_merged[df_merged['is_finetune'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()

# Print results
print(f"Total rows: {len(dat)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Finetune rows: {len(finetune_df)}")

# Inspect the training set
print(train_df.head())

Total rows: 678013
Train rows: 500000
Test rows: 100000
Finetune rows: 76782
   IDpol  ClaimNb  Exposure  Area  Frequency  VehGas_'Regular'  VehBrand_B10  \
1      3      1.0      0.77   4.0   1.298701               1.0           0.0   
3     10      1.0      0.09   2.0  11.111111               0.0           0.0   
4     11      1.0      0.84   2.0   1.190476               0.0           0.0   
5     13      1.0      0.52   5.0   1.923077               1.0           0.0   
7     17      1.0      0.27   3.0   3.703704               0.0           0.0   

   VehBrand_B11  VehBrand_B12  VehBrand_B13  ...  DrivAgeGLM_41-50  \
1           0.0           1.0           0.0  ...               0.0   
3           0.0           1.0           0.0  ...               1.0   
4           0.0           1.0           0.0  ...               1.0   
5           0.0           1.0           0.0  ...               0.0   
7           0.0           1.0           0.0  ...               0.0   

   DrivAgeGLM_51-70  

In [4]:
train_df = train_df.drop(["Frequency", "IDpol"], axis = 1).astype('float32')
test_df = test_df.drop(["Frequency", "IDpol"], axis = 1).astype('float32')

In [5]:
# Separate Predictors from Meta Data
meta_cols = ['ClaimNb', 'Exposure']
pred_cols = train_df.columns.difference(meta_cols)

X_train = train_df[pred_cols]
meta_train = train_df[meta_cols]

X_test = test_df[pred_cols]
meta_test = test_df[meta_cols]

# Concatenate 
final_train = pd.concat([X_train, meta_train], axis=1)
final_test = pd.concat([X_test, meta_test], axis=1)

print(final_train.head())

   Area  BonusMalusGLM  DensityGLM  DrivAgeGLM_21-25  DrivAgeGLM_26-30  \
1   4.0           50.0    7.104144               0.0               0.0   
3   2.0           50.0    4.330733               0.0               0.0   
4   2.0           50.0    4.330733               0.0               0.0   
5   5.0           50.0    8.007367               0.0               0.0   
7   3.0           68.0    4.919981               0.0               0.0   

   DrivAgeGLM_31-40  DrivAgeGLM_41-50  DrivAgeGLM_51-70  DrivAgeGLM_71+  \
1               0.0               0.0               1.0             0.0   
3               0.0               1.0               0.0             0.0   
4               0.0               1.0               0.0             0.0   
5               1.0               0.0               0.0             0.0   
7               1.0               0.0               0.0             0.0   

   Region_R21  ...  VehBrand_B5  VehBrand_B6  VehGas_'Regular'  VehPowerGLM_5  \
1         0.0  ...     

In [ ]:
# ---------------------------------------------------------
# Load the Embeddings
# ---------------------------------------------------------
# Load the NPZ files (ensure the paths match your directory structure)
train_emb_data = np.load("embeddings/gemini_train_embeddings.npz")
test_emb_data = np.load("embeddings/gemini_test_embeddings.npz")

# Convert the 'X' matrix (embeddings) into DataFrames
# We use the index from the raw dataframes (train_df/test_df) to ensure perfect alignment
train_emb_df = pd.DataFrame(train_emb_data['X'], index=train_df.index).add_prefix('emb_')
test_emb_df = pd.DataFrame(test_emb_data['X'], index=test_df.index).add_prefix('emb_')

In [7]:
# ---------------------------------------------------------
# PCA for Dimensionality Reduction 
# ---------------------------------------------------------
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
pca = PCA(n_components=48)

# Fit on training embeddings and transform both sets
train_emb_reduced = pca.fit_transform(scaler.fit_transform(train_emb_df))
test_emb_reduced = pca.transform(scaler.transform(test_emb_df))

# Convert back to DataFrame
train_pca_df = pd.DataFrame(train_emb_reduced, index=train_df.index).add_prefix('PC_')
test_pca_df = pd.DataFrame(test_emb_reduced, index=test_df.index).add_prefix('PC_')

# ---------------------------------------------------------
# Combine Raw Features + Embedding PCA
# ---------------------------------------------------------
# This joins your engineered raw columns with the embedding components
X_train_combined = pd.concat([X_train, train_pca_df], axis=1)
X_test_combined = pd.concat([X_test, test_pca_df], axis=1)

print(f"Combined Feature Count: {X_train_combined.shape[1]}")

Combined Feature Count: 96


In [8]:
# ---------------------------------------------------------
# Prepare Training Data
# ---------------------------------------------------------
# Identify predictor columns (The PCs)
# We exclude 'response' and 'offset' to get just the X matrix
predictors = [c for c in final_train.columns if c not in ['ClaimNb', 'Exposure']]

X_train = final_train[predictors]
y_train = final_train['ClaimNb']

# Statsmodels does not add an intercept by default.
# PCA centers data, but you still need an intercept for the baseline rate.
X_train = sm.add_constant(X_train_combined)

# Define the offset for training
train_offset = np.log(final_train['Exposure'])

# ---------------------------------------------------------
# Fit the Poisson GLM
# ---------------------------------------------------------
# Instantiate the model
glm_model = sm.GLM(
    endog=y_train, 
    exog=X_train, 
    offset=train_offset,
    family=sm.families.Poisson()
)

# Fit the model
results = glm_model.fit()

print(results.summary())

# ---------------------------------------------------------
# Predict on Test Data
# ---------------------------------------------------------
X_test = final_test[predictors]
X_test = sm.add_constant(X_test_combined, has_constant='add') # Ensure structure matches Train

# Define offset for testing
test_offset = np.log(final_test['Exposure'])

# Generate Predictions (Expected Counts)
# We pass the test offset here so predictions are scaled correctly
predictions = results.predict(exog=X_test, offset=test_offset)

# Add predictions back to the test dataframe for analysis
final_test['predicted_counts'] = predictions

print("\nFirst 5 Predictions:")
print(final_test[['ClaimNb', 'Exposure', 'predicted_counts']].head())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               500000
Model:                            GLM   Df Residuals:                   499903
Model Family:                 Poisson   Df Model:                           96
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0341e+05
Date:                Fri, 08 May 2026   Deviance:                   1.5602e+05
Time:                        10:45:28   Pearson chi2:                 1.14e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01627
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -2.1709      0.271  

In [9]:
# Access the Poisson family object from your fitted model
poisson_family = results.family

# Calculate Total Deviance for the Test Set
# Inputs: (Endog/Actual, Mu/Predicted)
test_deviance_total = poisson_family.deviance(
    final_test['ClaimNb'], 
    final_test['predicted_counts']
)

# Calculate Mean Deviance
test_mean_deviance = test_deviance_total / len(final_test)

print(f"Test Mean Poisson Deviance: {test_mean_deviance:.4f}")

Test Mean Poisson Deviance: 0.3168


In [10]:
import gc

# Define Grids
K_values = [4, 8, 12, 24, 48, 96]
train_sizes = [500, 1000, 2000, 5000, 10000, 20000, 40000, 80000]

# Initialize storage
results_grid = {k: [] for k in K_values}

# Preparation
# We start with the RAW embeddings and RAW features (not yet transformed)
# Ensure train_emb_df and test_emb_df are loaded as per previous steps
raw_pred_cols = list(train_df.columns.difference(['ClaimNb', 'Exposure']))

# Shuffle the raw training data once to ensure randomization
# We must shuffle the raw features and embeddings together
indices = np.random.permutation(len(train_df))
train_df_shuffled = train_df.iloc[indices].reset_index(drop=True)
train_emb_shuffled = train_emb_df.iloc[indices].reset_index(drop=True)

# Outer Loop: Training Size
for size in train_sizes:
    print(f"Processing Training Size: {size}...")
    
    # Slice raw data for this iteration
    X_raw_train_sub = train_emb_shuffled.iloc[:size]
    y_train_sub = train_df_shuffled.iloc[:size]['ClaimNb']
    off_train_sub = np.log(train_df_shuffled.iloc[:size]['Exposure'])
    
    # Get raw features for this iteration
    X_feat_train_sub = train_df_shuffled.iloc[:size][raw_pred_cols]
    X_feat_test = test_df[raw_pred_cols]
    
    # Fit PCA only on the current subset
    scaler = StandardScaler()
    pca = PCA(n_components=max(K_values), random_state=42)
    
    # Fit on training subset, transform both train and test
    X_train_pca_all = pca.fit_transform(scaler.fit_transform(X_raw_train_sub))
    X_test_pca_all = pca.transform(scaler.transform(test_emb_df))
    
    # Inner Loop: Complexity (K)
    for k in K_values:
        # Slice the first k components from the PCA fitted for this size
        pc_cols_train = X_train_pca_all[:, :k]
        pc_cols_test = X_test_pca_all[:, :k]
        
        # Combine Raw Features + PCA Components
        # We use numpy concatenation for speed inside the loop
        X_train_k = np.hstack([X_feat_train_sub.values, pc_cols_train])
        X_test_k = np.hstack([X_feat_test.values, pc_cols_test])
        
        # Add constant for statsmodels
        X_train_k = sm.add_constant(X_train_k)
        X_test_k = sm.add_constant(X_test_k, has_constant='add')
        
        try:
            # Fit Poisson GLM
            model = sm.GLM(
                endog=y_train_sub, 
                exog=X_train_k, 
                offset=off_train_sub, 
                family=sm.families.Poisson()
            ).fit()
            
            # Predict and Score
            preds = model.predict(X_test_k, offset=np.log(test_df['Exposure']))
            mpd = mean_poisson_deviance(test_df['ClaimNb'], preds)
            results_grid[k].append(mpd)
        except Exception as e:
            results_grid[k].append(None)
            
    # Cleanup memory for large datasets
    del X_train_pca_all, X_test_pca_all
    gc.collect()

# Results Table
df_results = pd.DataFrame(results_grid, index=train_sizes)
df_results.index.name = "TrainSize"
print("\n--- Leakage-Free Deviance Table ---")
print(df_results)

Processing Training Size: 500...
Processing Training Size: 1000...
Processing Training Size: 2000...
Processing Training Size: 5000...
Processing Training Size: 10000...
Processing Training Size: 20000...
Processing Training Size: 40000...
Processing Training Size: 80000...

--- Leakage-Free Deviance Table ---
                   4            8             12           24            48  \
TrainSize                                                                     
500        124.628634  1508.768513  43324.474762  3779.709016  5.402938e+20   
1000         0.942467     0.768867      0.741340     0.762256  8.172966e-01   
2000         0.428137     0.429408      0.432307     0.439711  4.629748e-01   
5000         0.378089     0.379817      0.380814     0.383305  3.863514e-01   
10000        0.325355     0.325351      0.326206     0.325510  3.238212e-01   
20000        0.323081     0.323003      0.323116     0.322434  3.200051e-01   
40000        0.322181     0.321951      0.321971     0.3

In [11]:
df_results

,4,8,12,24,48,96
TrainSize,,,,,,
500,124.628634,1508.768513,43324.474762,3779.709016,5.402938e+20,2.223539e+46
1000,0.942467,0.768867,0.741340,0.762256,8.172966e-01,3.220999e+01
2000,0.428137,0.429408,0.432307,0.439711,4.629748e-01,5.439409e-01
5000,0.378089,0.379817,0.380814,0.383305,3.863514e-01,3.967287e-01
10000,0.325355,0.325351,0.326206,0.325510,3.238212e-01,3.256630e-01
20000,0.323081,0.323003,0.323116,0.322434,3.200051e-01,3.198021e-01
40000,0.322181,0.321951,0.321971,0.321252,3.185360e-01,3.174488e-01
80000,0.321753,0.321498,0.321555,0.320612,3.177238e-01,3.161832e-01
